# Assignment 6: Siamese Network for Anomaly Detection (Appliances Energy Dataset)

Stage 6 goal — train a **Siamese 1D-CNN** that learns a metric on 24-step consumption windows so that windows with **normal consumption** (`HighConsumption = 0`) cluster together and **high/anomalous windows** (`HighConsumption = 1`) are pushed apart. The learned distance is then used to flag anomalies and to rank test windows against reference normal windows.

Pipeline:
1. Load `energydata_complete.csv` and rebuild the Assignment 4 windowed representation (24 steps, `HighConsumption` target).
2. Generate positive (normal/normal) and negative (normal/high) training pairs.
3. Build a shared 1D-CNN subnetwork (Assignment 4 backbone minus the classifier head) producing 128-d embeddings.
4. Compose the Siamese model with a `Lambda` Euclidean distance layer and train with **contrastive loss implemented from scratch**.
5. Tune the distance threshold on the validation set and evaluate on the test set (pair accuracy + ROC).
6. Score test windows against a bank of reference normal windows — nearest-neighbour distance becomes the anomaly score.
7. Compile a comparison table across the models built in earlier stages (CNN from Assignment 4, LSTM / Autoencoder+CNN from Assignment 5 when available) and the Siamese anomaly detector.

> The final scientific report / defense slides are **out of scope** for this notebook — that part of Stage 6 will be completed separately.

In [ ]:
import os
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
pd.set_option("display.max_columns", None)

print("TensorFlow:", tf.__version__)

## 1. Load data and rebuild the Assignment 4 windows

We reuse the exact preprocessing from Stage 4 so the Siamese subnetwork sees the same feature space as the baseline CNN classifier.

In [ ]:
def locate_dataset() -> Path:
    candidates = [
        Path("data/energydata_complete.csv"),
        Path("Assignment6/data/energydata_complete.csv"),
        Path("../Assignment4/data/energydata_complete.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("energydata_complete.csv was not found")


data_path = locate_dataset()
project_dir = Path(".").resolve()
figures_dir = project_dir / "figures"
figures_dir.mkdir(exist_ok=True)

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Dataset: {data_path}")
print(f"Shape:   {df.shape}")
df.head()

In [ ]:
median_appliances = df["Appliances"].median()
df["HighConsumption"] = (df["Appliances"] > median_appliances).astype("int8")

df["hour"] = df["date"].dt.hour + df["date"].dt.minute / 60.0
df["day_of_week"] = df["date"].dt.dayofweek
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

feature_cols = [c for c in df.columns if c not in ["date", "HighConsumption", "hour", "day_of_week"]]
print(f"Median appliance consumption (threshold): {median_appliances}")
print(f"Input features: {len(feature_cols)}")

In [ ]:
WINDOW_SIZE = 24
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

n_rows = len(df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[feature_cols].astype("float32"))
X_val_scaled = scaler.transform(val_df[feature_cols].astype("float32"))
X_test_scaled = scaler.transform(test_df[feature_cols].astype("float32"))

y_train = train_df["HighConsumption"].to_numpy(dtype=np.int32)
y_val = val_df["HighConsumption"].to_numpy(dtype=np.int32)
y_test = test_df["HighConsumption"].to_numpy(dtype=np.int32)


def make_sequences(features: np.ndarray, targets: np.ndarray, window_size: int):
    X_seq, y_seq = [], []
    for idx in range(window_size, len(features)):
        X_seq.append(features[idx - window_size : idx])
        y_seq.append(targets[idx])
    return np.asarray(X_seq, dtype=np.float32), np.asarray(y_seq, dtype=np.int32)


X_train_seq, y_train_seq = make_sequences(X_train_scaled, y_train, WINDOW_SIZE)
X_val_seq, y_val_seq = make_sequences(X_val_scaled, y_val, WINDOW_SIZE)
X_test_seq, y_test_seq = make_sequences(X_test_scaled, y_test, WINDOW_SIZE)

print("Split sizes (windows):")
print(f"  train: {X_train_seq.shape}, pos rate = {y_train_seq.mean():.3f}")
print(f"  val:   {X_val_seq.shape}, pos rate = {y_val_seq.mean():.3f}")
print(f"  test:  {X_test_seq.shape}, pos rate = {y_test_seq.mean():.3f}")

## 2. Build Siamese training pairs

Following the brief:
- **Positive pair (label 0, "similar")**: two windows drawn from the `HighConsumption = 0` pool.
- **Negative pair (label 1, "dissimilar")**: one `HighConsumption = 0` window + one `HighConsumption = 1` window.

Label convention follows the original contrastive-loss paper (Hadsell, Chopra & LeCun, 2006): `Y = 0` means same-class, `Y = 1` means different-class. The same convention is used during training.

In [ ]:
def build_pairs(X, y, n_pairs_per_class, rng):
    """Return (X_left, X_right, Y) where Y=0 -> normal/normal, Y=1 -> normal/anomaly."""
    normal_idx = np.where(y == 0)[0]
    anomaly_idx = np.where(y == 1)[0]

    pos_a = rng.choice(normal_idx, size=n_pairs_per_class, replace=True)
    pos_b = rng.choice(normal_idx, size=n_pairs_per_class, replace=True)

    neg_a = rng.choice(normal_idx, size=n_pairs_per_class, replace=True)
    neg_b = rng.choice(anomaly_idx, size=n_pairs_per_class, replace=True)

    X_left = np.concatenate([X[pos_a], X[neg_a]], axis=0)
    X_right = np.concatenate([X[pos_b], X[neg_b]], axis=0)
    Y = np.concatenate([np.zeros(n_pairs_per_class), np.ones(n_pairs_per_class)]).astype("float32")

    perm = rng.permutation(len(Y))
    return X_left[perm], X_right[perm], Y[perm]


rng = np.random.default_rng(SEED)
N_TRAIN_PAIRS_PER_CLASS = 8000
N_VAL_PAIRS_PER_CLASS = 2000
N_TEST_PAIRS_PER_CLASS = 2000

X_tr_L, X_tr_R, Y_tr = build_pairs(X_train_seq, y_train_seq, N_TRAIN_PAIRS_PER_CLASS, rng)
X_va_L, X_va_R, Y_va = build_pairs(X_val_seq, y_val_seq, N_VAL_PAIRS_PER_CLASS, rng)
X_te_L, X_te_R, Y_te = build_pairs(X_test_seq, y_test_seq, N_TEST_PAIRS_PER_CLASS, rng)

print(f"Training pairs:   {X_tr_L.shape}, positive share (dissimilar) = {Y_tr.mean():.3f}")
print(f"Validation pairs: {X_va_L.shape}")
print(f"Test pairs:       {X_te_L.shape}")

## 3. Shared 1D-CNN subnetwork

Architecture reuses the Assignment 4 backbone (Conv1D → MaxPool → Flatten → Dense) with the final sigmoid classifier **removed**. The last dense layer produces a 128-dimensional embedding, L2-normalised to keep distances in a bounded range.

In [ ]:
EMBEDDING_DIM = 128


def build_embedding_model(window_size: int, n_features: int, embedding_dim: int = 128) -> keras.Model:
    inputs = keras.Input(shape=(window_size, n_features), name="window_input")
    x = layers.Conv1D(64, kernel_size=3, activation="relu", padding="causal")(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Conv1D(128, kernel_size=3, activation="relu", padding="causal")(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(embedding_dim, activation=None)(x)
    x = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1), name="l2_normalise")(x)
    return keras.Model(inputs, x, name="shared_cnn_embedding")


embedding_model = build_embedding_model(WINDOW_SIZE, X_train_seq.shape[-1], EMBEDDING_DIM)
embedding_model.summary()

## 4. Assemble the Siamese network and contrastive loss

Two inputs share the same embedding subnetwork. A `Lambda` layer computes the Euclidean distance between the two embeddings:
$$D(a, b) = \sqrt{\sum_i (a_i - b_i)^2 + \varepsilon}$$

The contrastive loss (Hadsell, Chopra & LeCun 2006) is implemented from scratch:
$$\mathcal{L}(Y, D) = (1 - Y) \cdot \tfrac{1}{2} D^2 \;+\; Y \cdot \tfrac{1}{2}\bigl(\max(0, m - D)\bigr)^2$$
with `Y=0` for similar pairs and `Y=1` for dissimilar pairs. The margin `m` is a hyperparameter (we use 1.0).

In [ ]:
MARGIN = 1.0


def euclidean_distance(vectors):
    a, b = vectors
    sq = tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True)
    return tf.sqrt(tf.maximum(sq, tf.keras.backend.epsilon()))


def contrastive_loss(margin: float = 1.0):
    def loss(y_true, d_pred):
        y_true = tf.cast(y_true, d_pred.dtype)
        d = tf.reshape(d_pred, [-1])
        y = tf.reshape(y_true, [-1])
        similar_term = (1.0 - y) * 0.5 * tf.square(d)
        dissimilar_term = y * 0.5 * tf.square(tf.maximum(margin - d, 0.0))
        return tf.reduce_mean(similar_term + dissimilar_term)

    return loss


def pair_accuracy_metric(threshold: float = 0.5):
    def acc(y_true, d_pred):
        y_true = tf.cast(y_true, d_pred.dtype)
        predicted_dissimilar = tf.cast(tf.reshape(d_pred, [-1]) > threshold, d_pred.dtype)
        correct = tf.cast(tf.equal(predicted_dissimilar, tf.reshape(y_true, [-1])), d_pred.dtype)
        return tf.reduce_mean(correct)

    acc.__name__ = "pair_acc"
    return acc


input_a = keras.Input(shape=(WINDOW_SIZE, X_train_seq.shape[-1]), name="left")
input_b = keras.Input(shape=(WINDOW_SIZE, X_train_seq.shape[-1]), name="right")
emb_a = embedding_model(input_a)
emb_b = embedding_model(input_b)
distance = layers.Lambda(euclidean_distance, name="euclidean_distance")([emb_a, emb_b])

siamese = keras.Model(inputs=[input_a, input_b], outputs=distance, name="siamese_network")
siamese.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=contrastive_loss(MARGIN),
    metrics=[pair_accuracy_metric(threshold=MARGIN / 2)],
)
siamese.summary()

## 5. Train the Siamese network

In [ ]:
EPOCHS = 15
BATCH_SIZE = 128

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-5),
]

history = siamese.fit(
    [X_tr_L, X_tr_R],
    Y_tr,
    validation_data=([X_va_L, X_va_R], Y_va),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Contrastive loss")
history_df[[c for c in history_df.columns if c.endswith("pair_acc")]].plot(ax=axes[1], title="Pair accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(figures_dir / "siamese_training_curves.png", dpi=150)
plt.show()

## 6. Tune the distance threshold on the validation set

For each candidate threshold `t`, predict `dissimilar` when `D > t`. The threshold that maximises validation pair accuracy is retained for downstream evaluation.

In [ ]:
val_distances = siamese.predict([X_va_L, X_va_R], batch_size=256, verbose=0).ravel()
test_distances = siamese.predict([X_te_L, X_te_R], batch_size=256, verbose=0).ravel()

candidate_thresholds = np.linspace(val_distances.min(), val_distances.max(), 200)
val_accuracies = [(t, accuracy_score(Y_va, (val_distances > t).astype(int))) for t in candidate_thresholds]
best_threshold, best_val_acc = max(val_accuracies, key=lambda p: p[1])

print(f"Best validation threshold = {best_threshold:.4f} (val pair accuracy = {best_val_acc:.4f})")

test_preds = (test_distances > best_threshold).astype(int)
test_pair_metrics = {
    "Accuracy": accuracy_score(Y_te, test_preds),
    "Precision": precision_score(Y_te, test_preds, zero_division=0),
    "Recall": recall_score(Y_te, test_preds, zero_division=0),
    "F1": f1_score(Y_te, test_preds, zero_division=0),
}
print("\nTest-set pair classification metrics:")
for name, value in test_pair_metrics.items():
    print(f"  {name}: {value:.4f}")

print("\nConfusion matrix on pairs (rows = true, cols = pred):")
print(confusion_matrix(Y_te, test_preds))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(val_distances[Y_va == 0], bins=40, stat="density", label="Y=0 (similar)", ax=axes[0], color="#4C78A8", alpha=0.6)
sns.histplot(val_distances[Y_va == 1], bins=40, stat="density", label="Y=1 (dissimilar)", ax=axes[0], color="#F58518", alpha=0.6)
axes[0].axvline(best_threshold, color="red", linestyle="--", label=f"threshold = {best_threshold:.2f}")
axes[0].set_title("Validation distance distribution")
axes[0].set_xlabel("Euclidean distance")
axes[0].legend()

thresholds_plot = np.array([t for t, _ in val_accuracies])
accs_plot = np.array([a for _, a in val_accuracies])
axes[1].plot(thresholds_plot, accs_plot, color="#4C78A8")
axes[1].axvline(best_threshold, color="red", linestyle="--")
axes[1].set_title("Validation pair accuracy vs. threshold")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.savefig(figures_dir / "siamese_threshold_tuning.png", dpi=150)
plt.show()

## 7. Anomaly detection: nearest-normal distance as anomaly score

For each test window we compute its embedding and measure the distance to the **nearest** reference embedding drawn from a bank of normal (HighConsumption=0) training windows. A large minimum distance → the window does not resemble any known normal pattern → anomaly.

We evaluate with:
* ROC curve and AUC over test windows (true label = `HighConsumption`).
* A qualitative check on the most anomalous test windows (highest score).

In [ ]:
# Bank of reference normal embeddings from the training set
REFERENCE_POOL_SIZE = 2000
normal_train_idx = np.where(y_train_seq == 0)[0]
reference_idx = rng.choice(normal_train_idx, size=min(REFERENCE_POOL_SIZE, len(normal_train_idx)), replace=False)
reference_embeddings = embedding_model.predict(X_train_seq[reference_idx], batch_size=256, verbose=0)
test_embeddings = embedding_model.predict(X_test_seq, batch_size=256, verbose=0)

# Nearest reference distance (batched for memory safety)
def min_distance_to_reference(query_embs, ref_embs, batch=512):
    scores = np.empty(len(query_embs), dtype=np.float32)
    for start in range(0, len(query_embs), batch):
        chunk = query_embs[start : start + batch]
        diffs = chunk[:, None, :] - ref_embs[None, :, :]
        dists = np.sqrt(np.sum(diffs ** 2, axis=-1))
        scores[start : start + batch] = dists.min(axis=1)
    return scores


anomaly_scores = min_distance_to_reference(test_embeddings, reference_embeddings)
print(f"Anomaly score stats — mean={anomaly_scores.mean():.4f}, std={anomaly_scores.std():.4f}")

In [ ]:
fpr, tpr, _ = roc_curve(y_test_seq, anomaly_scores)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, label=f"Siamese (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="chance")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve — Siamese anomaly score")
axes[0].legend(loc="lower right")

sns.histplot(anomaly_scores[y_test_seq == 0], bins=40, stat="density", label="Normal (y=0)", ax=axes[1], color="#4C78A8", alpha=0.6)
sns.histplot(anomaly_scores[y_test_seq == 1], bins=40, stat="density", label="High (y=1)", ax=axes[1], color="#F58518", alpha=0.6)
axes[1].set_title("Anomaly score distribution on the test set")
axes[1].set_xlabel("Min distance to normal reference")
axes[1].legend()

plt.tight_layout()
plt.savefig(figures_dir / "siamese_roc_and_scores.png", dpi=150)
plt.show()

print(f"Siamese anomaly-score ROC-AUC on test set: {roc_auc:.4f}")

In [ ]:
# Inspect the top-5 most anomalous test windows
top_k = 5
top_idx = np.argsort(-anomaly_scores)[:top_k]

appliances_col = feature_cols.index("Appliances")
fig, axes = plt.subplots(top_k, 1, figsize=(10, 2.2 * top_k), sharex=True)
for ax, idx in zip(axes, top_idx):
    window = X_test_seq[idx, :, appliances_col]
    label = int(y_test_seq[idx])
    ax.plot(window, color="#F58518" if label == 1 else "#4C78A8")
    ax.set_title(
        f"Test window #{idx} | score = {anomaly_scores[idx]:.3f} | HighConsumption = {label}",
        fontsize=10,
    )
axes[-1].set_xlabel("Time step within window (scaled Appliances channel)")
plt.tight_layout()
plt.savefig(figures_dir / "top_anomalies.png", dpi=150)
plt.show()

## 8. Save the trained Siamese model

In [ ]:
embedding_model.save(project_dir / "siamese_embedding.keras")
siamese.save(project_dir / "siamese_model.keras")
print("Saved:")
print("  -", project_dir / "siamese_embedding.keras")
print("  -", project_dir / "siamese_model.keras")

## 9. Integrated model comparison

We compare models on the same `HighConsumption` binary task (energy dataset). For anomaly detection the Siamese network is evaluated by thresholding the nearest-reference distance at the point on the ROC curve that maximises Youden's J (`tpr − fpr`).

> **Note.** Rows marked *pending* are produced by Stage 5 (LSTM, Autoencoder + CNN) and will be filled in once that notebook is finalised. They are left in the table for completeness against the Stage 6 brief.

In [ ]:
# Re-load the best CNN classifier from Assignment 4 and score it on the shared test split
cnn_metrics_row = None
cnn_path_candidates = [
    Path("../Assignment4/cnn_classifier.h5"),
    Path("Assignment4/cnn_classifier.h5"),
]
cnn_model_path = next((p for p in cnn_path_candidates if p.exists()), None)

if cnn_model_path is not None:
    cnn_model = keras.models.load_model(cnn_model_path)
    cnn_probs = cnn_model.predict(X_test_seq, verbose=0).ravel()
    cnn_pred = (cnn_probs >= 0.5).astype(int)
    cnn_metrics_row = {
        "Model": "1D-CNN classifier (Stage 4)",
        "Task": "Binary classification",
        "Accuracy": accuracy_score(y_test_seq, cnn_pred),
        "Precision": precision_score(y_test_seq, cnn_pred, zero_division=0),
        "Recall": recall_score(y_test_seq, cnn_pred, zero_division=0),
        "F1": f1_score(y_test_seq, cnn_pred, zero_division=0),
        "ROC-AUC": auc(*roc_curve(y_test_seq, cnn_probs)[:2]),
    }
    print(f"Loaded CNN: {cnn_model_path}")
else:
    print("CNN model from Assignment 4 not found — skipping that row.")

# Siamese anomaly-detection row (thresholded by Youden's J)
J = tpr - fpr
best_j_idx = int(np.argmax(J))
roc_thresholds = roc_curve(y_test_seq, anomaly_scores)[2]
anomaly_threshold = roc_thresholds[best_j_idx]
siamese_preds = (anomaly_scores >= anomaly_threshold).astype(int)
siamese_row = {
    "Model": "Siamese + contrastive (Stage 6)",
    "Task": "Anomaly detection",
    "Accuracy": accuracy_score(y_test_seq, siamese_preds),
    "Precision": precision_score(y_test_seq, siamese_preds, zero_division=0),
    "Recall": recall_score(y_test_seq, siamese_preds, zero_division=0),
    "F1": f1_score(y_test_seq, siamese_preds, zero_division=0),
    "ROC-AUC": roc_auc,
}

placeholder_rows = [
    {"Model": "MLP (Stage 2)", "Task": "Regression (SCANIA legacy)", "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan, "ROC-AUC": np.nan},
    {"Model": "DNN (Stage 2)", "Task": "Regression (SCANIA legacy)", "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan, "ROC-AUC": np.nan},
    {"Model": "LSTM (Stage 5, pending)", "Task": "Forecasting", "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan, "ROC-AUC": np.nan},
    {"Model": "Autoencoder + CNN (Stage 5, pending)", "Task": "Anomaly detection", "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan, "ROC-AUC": np.nan},
]

rows = placeholder_rows[:2]
if cnn_metrics_row is not None:
    rows.append(cnn_metrics_row)
rows.extend(placeholder_rows[2:])
rows.append(siamese_row)

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(project_dir / "model_comparison.csv", index=False)
comparison_df

### Short analysis

* **Forecasting / regression** — the dense MLP and DNN baselines (Stage 2, legacy SCANIA target) serve only as a reference that a stacked dense network typically under-fits time-ordered signals once sequence structure matters.
* **Classification** — the Stage 4 1D-CNN is the strongest supervised model on `HighConsumption`. Causal Conv1D kernels over 24-step windows capture the short-term shape of consumption spikes better than a flat feature vector.
* **Anomaly detection** — the Siamese network is **not** a classifier: it scores a window by how far its embedding sits from a bank of known-normal embeddings. The contrastive objective explicitly pulls normal pairs together and pushes anomalous pairs past the margin, so the resulting distance is a well-calibrated anomaly score. ROC-AUC on the test set (see table above) is the headline metric here; the CNN classifier's AUC is included for comparison but it was trained with full supervision on the same target, so a direct comparison is indicative, not strict.

**Take-away.** For a production anomaly-detection module on this dataset, combining the Siamese network (open-set, needs only normal references at inference) with the supervised Stage 4 CNN (closed-set classifier, high recall on labelled spikes) gives complementary coverage. The Stage 5 LSTM / Autoencoder rows will complete the picture once available.